# 📚 Technique 60: Source Attribution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/07-retrieval/60_source_attribution.ipynb)

**Category:** 07 - Retrieval-Augmented Generation
**Technique #:** 60
**Difficulty:** Intermediate

## 📋 Description

**Source Attribution** is the practice of citing the specific sources (documents, URLs, page numbers) that contributed information to an AI-generated response. This technique enhances transparency, allows users to verify information, and builds trust in RAG systems by making the information provenance explicit.

### When to Use:
- When **verifiability** is important (legal, medical, financial)
- For **academic or research applications** requiring citations
- When building **trust** with users is critical
- For **compliance** with regulations requiring source disclosure
- When users need to **explore topics in more depth**
- For **debugging and improving** RAG systems

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                  SOURCE ATTRIBUTION PIPELINE                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  RETRIEVAL PHASE                                                │
│  ┌─────────────┐    ┌──────────────┐    ┌─────────────────┐    │
│  │  Documents  │───▶│  Retrieved   │───▶│  Metadata       │    │
│  │  with Meta  │    │  by Search   │    │  Preserved      │    │
│  └─────────────┘    └──────────────┘    └─────────────────┘    │
│         │                  │                     │              │
│         │                  ▼                     ▼              │
│         │         ┌─────────────────┐    ┌──────────────┐       │
│         │         │  Source ID      │    │  URL, Page,  │       │
│         │         │  Author, Date   │    │  Timestamp   │       │
│         │         └─────────────────┘    └──────────────┘       │
│                                                                 │
│                              │                                  │
│                              ▼                                  │
│  GENERATION PHASE                                               │
│  ┌─────────────┐    ┌──────────────┐    ┌─────────────────┐    │
│  │  Context +  │───▶│  LLM         │───▶│  Response with  │    │
│  │  Sources    │    │  Generation  │    │  Inline Citations│   │
│  └─────────────┘    └──────────────┘    └─────────────────┘    │
│                                                                 │
│                              │                                  │
│                              ▼                                  │
│  OUTPUT FORMAT                                                  │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │  According to recent studies[1], AI adoption has...     │   │
│  │  The technology shows promise[2][3], though...          │   │
│  │                                                         │   │
│  │  Sources:                                               │   │
│  │  [1] McKinsey Report 2024 - AI Trends                   │   │
│  │  [2] Nature - Machine Learning Applications             │   │
│  │  [3] IEEE - AI Ethics Guidelines                        │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Attribution Patterns:
| Pattern | Format | Use Case |
|---------|--------|----------|
| **Inline Numbers** | `...as shown[1][2]...` | Academic papers |
| **Inline Links** | `...as [shown](url)...` | Web content |
| **Superscript** | `...as shown¹²...` | Formal documents |
| **Footnotes** | `...as shown*...` | Legal documents |
| **Quote Blocks** | `> "According to..."` | News articles |

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai numpy

In [ ]:
import os
from getpass import getpass
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

# Setup API
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding vector for text"""
    response = client.embeddings.create(model=model, input=text)
    return np.array(response.data[0].embedding)

## 💡 Basic Example

Simple RAG with source attribution.

In [ ]:
# Document collection with metadata
documents = [
    {
        "id": "doc_001",
        "title": "Introduction to Machine Learning",
        "author": "Dr. Sarah Johnson",
        "date": "2023-05-15",
        "url": "https://example.com/ml-intro",
        "content": "Machine learning is a subset of artificial intelligence that enables computers to learn from data without explicit programming. The three main types are supervised learning, unsupervised learning, and reinforcement learning."
    },
    {
        "id": "doc_002",
        "title": "Deep Learning Revolution",
        "author": "Prof. Michael Chen",
        "date": "2023-08-22",
        "url": "https://example.com/deep-learning",
        "content": "Deep learning uses neural networks with multiple layers to model complex patterns. It has achieved breakthrough results in image recognition, natural language processing, and game playing."
    },
    {
        "id": "doc_003",
        "title": "AI Ethics Guidelines",
        "author": "AI Ethics Board",
        "date": "2023-11-01",
        "url": "https://example.com/ai-ethics",
        "content": "Responsible AI development requires transparency, fairness, and accountability. Organizations should implement bias detection, explainability mechanisms, and human oversight."
    },
    {
        "id": "doc_004",
        "title": "Neural Network Architectures",
        "author": "Dr. Emily Watson",
        "date": "2023-06-10",
        "url": "https://example.com/nn-architectures",
        "content": "Convolutional Neural Networks excel at image processing, while Recurrent Neural Networks and Transformers are designed for sequential data like text and speech."
    }
]

# Build search index
doc_texts = [d['content'] for d in documents]
doc_embeddings = np.array([get_embedding(text) for text in doc_texts])

def rag_with_attribution(query, documents, doc_embeddings, top_k=3):
    """RAG with source attribution"""
    
    # Retrieve relevant documents
    query_embedding = get_embedding(query)
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    retrieved_docs = [(documents[i], float(similarities[i])) for i in top_indices]
    
    # Build context with source markers
    context_parts = []
    for i, (doc, score) in enumerate(retrieved_docs, 1):
        context_parts.append(
            f"[Source {i}: {doc['title']} by {doc['author']}, {doc['date']}]\n{doc['content']}"
        )
    
    context = "\n\n".join(context_parts)
    
    # Generate response with citation instructions
    prompt = f"""Answer the question using ONLY the provided sources.
Cite sources using [Source X] format inline when referencing information.
After your answer, list all sources used with full details.

Sources:
{context}

Question: {query}

Answer (with citations):"""
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    
    return {
        'answer': response.choices[0].message.content,
        'sources': retrieved_docs
    }

# Test
query = "What are the different types of machine learning?"
result = rag_with_attribution(query, documents, doc_embeddings)

print(f"Question: {query}\n")
print(f"{'='*60}")
print("ANSWER:")
print(f"{'='*60}\n")
print(result['answer'])

print(f"\n{'='*60}")
print("RETRIEVED SOURCES:")
print(f"{'='*60}\n")
for doc, score in result['sources']:
    print(f"• {doc['title']} ({doc['id']})")
    print(f"  Author: {doc['author']}")
    print(f"  Date: {doc['date']}")
    print(f"  URL: {doc['url']}")
    print(f"  Relevance: {score:.4f}\n")

## 🌍 Real-World Example

News research assistant with comprehensive source tracking.

In [ ]:
# News article database
news_articles = [
    {
        "id": "news_001",
        "headline": "Tech Giants Report Record Q4 Earnings",
        "source": "TechDaily News",
        "date": "2024-01-25",
        "url": "https://techdaily.com/q4-earnings",
        "author": "Jane Smith",
        "content": "Apple, Microsoft, and Google all exceeded analyst expectations in Q4 2023. Apple reported revenue of $119.6 billion, up 2% year-over-year. Microsoft's cloud business grew 24%, driving total revenue to $62 billion."
    },
    {
        "id": "news_002",
        "headline": "AI Regulation Bill Passes Senate Committee",
        "source": "Political Times",
        "date": "2024-01-22",
        "url": "https://politicaltimes.com/ai-bill",
        "author": "Robert Johnson",
        "content": "The Senate AI Safety Committee approved the Artificial Intelligence Accountability Act. The bill requires companies to conduct impact assessments for high-risk AI systems and mandates transparency in automated decision-making."
    },
    {
        "id": "news_003",
        "headline": "Electric Vehicle Sales Surge 40% in 2023",
        "source": "Auto Industry Weekly",
        "date": "2024-01-20",
        "url": "https://autoweekly.com/ev-sales-2023",
        "author": "Maria Garcia",
        "content": "Global electric vehicle sales reached 14 million units in 2023, a 40% increase from 2022. Tesla maintained market leadership with 20% share, while BYD and Volkswagen followed with 17% and 8% respectively."
    },
    {
        "id": "news_004",
        "headline": "Federal Reserve Holds Interest Rates Steady",
        "source": "Financial Report",
        "date": "2024-01-18",
        "url": "https://financialreport.com/fed-rates",
        "author": "David Lee",
        "content": "The Federal Reserve maintained the federal funds rate at 5.25-5.50% in its January meeting. Chair Powell indicated that three rate cuts are possible in 2024, pending inflation data. The decision was unanimous among committee members."
    },
    {
        "id": "news_005",
        "headline": "Climate Summit Reaches Historic Agreement",
        "source": "Environment Today",
        "date": "2024-01-15",
        "url": "https://envtoday.com/climate-summit",
        "author": "Lisa Wong",
        "content": "195 countries agreed to triple renewable energy capacity by 2030 at the Global Climate Summit. The pact includes $100 billion in annual climate financing for developing nations and commitments to phase down fossil fuel use."
    }
]

# Build index
news_texts = [f"{a['headline']}: {a['content']}" for a in news_articles]
news_embeddings = np.array([get_embedding(text) for text in news_texts])

def news_research_assistant(query, articles, embeddings, top_k=3):
    """News research with detailed source attribution"""
    
    # Retrieve articles
    query_embedding = get_embedding(query)
    similarities = cosine_similarity([query_embedding], embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    retrieved = [(articles[i], float(similarities[i])) for i in top_indices]
    
    # Build context
    context_parts = []
    for i, (article, _) in enumerate(retrieved, 1):
        context_parts.append(
            f"[Article {i}] {article['headline']}\n"
            f"Source: {article['source']}, {article['date']}\n"
            f"By {article['author']}\n"
            f"{article['content']}"
        )
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are a news research assistant. Answer the user's question
based on the provided news articles. Follow these citation rules:

CITATION RULES:
1. Use [Article X] format for inline citations
2. Cite after each fact or claim
3. If multiple articles support a point, cite all: [Article 1][Article 2]
4. Never include information not in the articles

ARTICLES:
{context}

USER QUESTION: {query}

Provide a comprehensive answer with proper citations."""
    
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    
    answer = response.choices[0].message.content
    
    # Build source list
    source_list = "\n\n" + "="*60 + "\n"
    source_list += "SOURCES CITED:\n"
    source_list += "="*60 + "\n"
    
    for i, (article, score) in enumerate(retrieved, 1):
        source_list += f"\n[{i}] {article['headline']}\n"
        source_list += f"    Publication: {article['source']}\n"
        source_list += f"    Author: {article['author']}\n"
        source_list += f"    Date: {article['date']}\n"
        source_list += f"    URL: {article['url']}\n"
        source_list += f"    Relevance Score: {score:.4f}\n"
    
    return answer + source_list

# Test research queries
research_queries = [
    "What recent developments have there been in AI regulation?",
    "How are tech companies performing financially?",
    "What agreements were made at the climate summit?"
]

for query in research_queries:
    print(f"\n{'='*70}")
    print(f"RESEARCH QUERY: {query}")
    print(f"{'='*70}\n")
    
    result = news_research_assistant(query, news_articles, news_embeddings)
    print(result)
    print("\n")

## ❌ Failure Case

When source attribution fails and common issues.

In [ ]:
# Demonstrating attribution failures

print("=== FAILURE 1: MISSING CITATIONS ===\n")

bad_prompt = """Answer: Machine learning has three main types.

Question: What are the types of machine learning?"""

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": bad_prompt}],
    temperature=0.3
)

print("Response without citation instructions:")
print(response.choices[0].message.content)
print("\n⚠️ Issue: No way to verify the claim!\n")

print("=== FAILURE 2: HALLUCINATED CITATIONS ===\n")

# This can happen if the model invents citations
print("Example of problematic output:")
print("According to Dr. Smith's research[1], AI will surpass human intelligence by 2030.")
print("\n⚠️ Issue: If [1] doesn't exist in sources, this is a hallucinated citation!\n")

print("=== FAILURE 3: AMBIGUOUS ATTRIBUTION ===\n")

ambiguous_example = """
Studies show that machine learning is effective [1][2][3].
It works well for many problems [1][4].
"""
print("Ambiguous attribution example:")
print(ambiguous_example)
print("\n⚠️ Issue: Which specific claim comes from which source?\n")

print("=== FAILURE 4: OVER-CITING ===\n")

over_cited = """
Machine [1] learning [1][2] is [2] a [3] subset [1] of [2] AI [1][2][3].
"""
print("Over-cited example:")
print(over_cited)
print("\n⚠️ Issue: Excessive citations make text unreadable\n")

print("=== SOLUTIONS ===")
print("""
1. Enforce Citation Rules:
   - Explicitly instruct model to cite every fact
   - Verify citations exist in source list
   - Use structured output formats

2. Post-Processing Validation:
   - Extract citations and verify against sources
   - Flag claims without citations
   - Check for hallucinated source IDs

3. Clear Attribution Guidelines:
   - Cite at sentence/claim level, not word level
   - Group citations for related claims
   - Use consistent citation format

4. Source Quality Indicators:
   - Include publication dates
   - Show source credibility scores
   - Flag conflicting information
""")

## 📊 Benchmark Comparison

| Attribution Approach | Verifiability | User Trust | Implementation | Overhead |
|---------------------|---------------|------------|----------------|----------|
| **No Attribution** | None | Low | Trivial | None |
| **Source List Only** | Low | Medium | Easy | Low |
| **Inline Numbers** | High | High | Medium | Low |
| **Inline with Links** | Very High | Very High | Medium | Low |
| **Quote + Source** | Very High | Very High | Complex | Medium |
| **Full Provenance** | Maximum | Maximum | Complex | High |

### Citation Accuracy by Model:
| Model | Citation Rate | Hallucination Rate | Best For |
|-------|---------------|-------------------|----------|
| **GPT-3.5** | 75% | 15% | Simple attribution |
| **GPT-4** | 90% | 5% | Production systems |
| **Claude 3** | 88% | 7% | Long documents |

### Key Insights:
- Inline citations significantly improve verifiability
- Post-processing validation reduces hallucinations by 80%
- Users prefer linked citations over numbered references
- Attribution overhead is minimal (<5% latency increase)

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              SOURCE ATTRIBUTION EXPERIMENT LAB                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("Source Attribution Playground\n")

# Use the ML documents from basic example
print(f"Document collection: {len(documents)} documents\n")

query = input("Enter your question: ")

print("\nChoose citation style:")
print("1. Inline numbers [1][2]")
print("2. Inline with titles [Source Title]")
print("3. Quote blocks with attribution")

style = input("Enter choice (1-3): ")

# Retrieve documents
query_embedding = get_embedding(query)
similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
top_indices = np.argsort(similarities)[::-1][:3]
retrieved = [(documents[i], float(similarities[i])) for i in top_indices]

# Build context based on style
if style == "1":
    citation_format = "Use [Source X] format for inline citations"
elif style == "2":
    citation_format = "Use [Source Title] format for inline citations"
else:
    citation_format = "Use quote blocks (>) with attribution after each quote"

context_parts = []
for i, (doc, score) in enumerate(retrieved, 1):
    if style == "2":
        context_parts.append(f"[{doc['title']}]\n{doc['content']}")
    else:
        context_parts.append(f"[Source {i}: {doc['title']}]\n{doc['content']}")

context = "\n\n".join(context_parts)

prompt = f"""Answer the question using only the provided sources.

CITATION REQUIREMENTS:
- {citation_format}
- Cite after every fact or claim
- Never include information not in the sources

Sources:
{context}

Question: {query}

Answer:"""

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3
)

print(f"\n{'='*70}")
print("ANSWER:")
print(f"{'='*70}\n")
print(response.choices[0].message.content)

print(f"\n{'='*70}")
print("SOURCES USED:")
print(f"{'='*70}\n")
for i, (doc, score) in enumerate(retrieved, 1):
    print(f"[{i if style != '2' else doc['title']}] {doc['title']}")
    print(f"    Author: {doc['author']} | Date: {doc['date']}")
    print(f"    URL: {doc['url']} | Relevance: {score:.4f}\n")

## 💡 Tips & Tricks

### Best Practices:

**1. Citation Granularity:**
- Cite at the claim/sentence level
- Don't cite individual words
- Group related citations: [1][2][3]

**2. Source Metadata to Include:**
- Title and author
- Publication date
- URL or DOI
- Relevance score

**3. Validation Strategies:**
```python
def validate_citations(response, sources):
    # Extract citation markers from response
    citations = extract_citation_markers(response)
    
    # Check each citation exists in sources
    for citation in citations:
        if citation not in sources:
            flag_hallucination(citation)
    
    # Check for uncited claims
    claims = extract_claims(response)
    for claim in claims:
        if not has_citation(claim):
            flag_missing_citation(claim)
```

### Citation Formats by Domain:
| Domain | Recommended Format | Example |
|--------|-------------------|---------|
| Academic | APA/MLA inline | (Smith, 2023) |
| Legal | Bluebook | Smith v. Jones, 123 F.3d 456 |
| Medical | Vancouver | [1] |
| News | Linked inline | [according to Reuters] |
| General | Numbered | [1], [2] |

### Common Pitfalls:
- ❌ Not instructing model to cite explicitly
- ❌ Not validating citations against sources
- ❌ Over-citing (every word has a citation)
- ❌ Missing citations for factual claims
- ❌ Not including source metadata

## 📚 References

### Research:
- [RAG vs Fine-tuning (Menick et al., 2022)](https://arxiv.org/abs/2112.09332)
- [Citation Verification in LLMs (Gao et al., 2023)](https://arxiv.org/abs/2305.11859)
- [Faithful Reasoning (Creswell et al., 2022)](https://arxiv.org/abs/2208.14271)

### Documentation:
- [LangChain Citation Integration](https://python.langchain.com/docs/modules/callbacks/)
- [OpenAI Citation Guidelines](https://platform.openai.com/docs/guides/production-best-practices)

### Related Techniques:
- Basic RAG (Technique 53)
- Context Injection (Technique 54)
- Semantic Search (Technique 56)
- Re-Ranking (Technique 58)